échauffement : construire une URL avec des paramètres

voici la f-string qui construit l'URL demandé :
f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom}"

In [3]:
num,typ,nom=3,"rue","tourelles"
f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom}"

'https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles'

échauffement : sélectionner des lignes avec iloc et loc

In [4]:
import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

1.sub contient les éléments de df qui sont strictement plus grand que 10. Soit :
pd.DataFrame({
    "number": [ 18, 12,],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
son index est [1,3]
2.df.iloc[:2] renvoie 
pd.DataFrame({
    "number": [3, 18],
    "name": ["tourelles", "BERNARDINS"],
})

df.loc[:2 ] renvoie
pd.DataFrame({
    "number": [3, 18, 7],
    "name": ["tourelles", "BERNARDINS", "Bouvart"],
})
La borne 2 est incluse par loc et exclue par iloc.
3.sub.loc[:2] renvoie
df = pd.DataFrame({
    "number": [3],
    "name": ["tourelles"],
})

sub.iloc[:2] renvoie
df = pd.DataFrame({
    "number": [ 18, 12],
    "name": [ "BERNARDINS", "Rivoli"],
})

4.sub.iloc[0] renvoie la colonne
number            18
name      BERNARDINS
Name: 1, dtype: object

sub.loc[0] renvoie une erreur, puisqu'il n'y a pas d'index valant 0 dans sub.

5. Avec iloc :
    sub.iloc[0]

Avec loc :
    sub.loc[1]

géolocalisation

In [5]:
# for starters, we only need the 'regular fit' pandas

import pandas as pd
# load the data in data/addresses.csv
# and display a few first lines

addresses = pd.read_csv('data/addresses.csv')
addresses.head(4)
# produce a small extract into data/addresses-small.csv
extract=addresses.head(10).to_csv('data/addresses-small.csv',index=False)

In [6]:
# requests is the swiss knife for doing http

import requests# votre code

def localize_one(num, typ, nom):
    urll=f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1"
    r= requests.get(urll)
    if not(r.status_code in [200,300]):
        print("Hum... il y a eu un problème dans la requête.")
        print(r)
        return None
    else:
        return r.json()


In [7]:
# and here is how we could use it
localize_one(18, 'rue', 'BERNARDINS')


{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]},
   'properties': {'label': '18 Rue des Bernardins 75005 Paris',
    'score': 0.7544495104895104,
    'housenumber': '18',
    'id': '75105_0896_00018',
    'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02',
    'name': '18 Rue des Bernardins',
    'postcode': '75005',
    'citycode': '75105',
    'x': 652355.3,
    'y': 6861340.65,
    'city': 'Paris',
    'district': 'Paris 5e Arrondissement',
    'context': '75, Paris, Île-de-France',
    'type': 'housenumber',
    'importance': 0.68356,
    'depcode': '75',
    'street': 'Rue des Bernardins',
    '_type': 'address'}}],
 'query': '18 rue BERNARDINS,Paris'}

In [8]:
# try to estimate how long it would take 
# to resolve 20_000 addresses this way
import time
t1=time.time()
localize_one(18, 'rue', 'BERNARDINS')
t2=time.time()
#On mesure l'heure avant, et l'heure après, et puis on l'affiche multipliée par 20 000.
print("Voici le temps que cela prendrait de faire les 30000 requêtes ainsi :")
print(20000*(t2-t1))
print("C'est bien long.")

Voici le temps que cela prendrait de faire les 30000 requêtes ainsi :
6180.028915405273
C'est bien long.


In [9]:
# requests is the swiss knife for doing http

import requests
# this one is to fake a file from a string
import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    # we don't care about the index for now
    # but note that if we want to merge the result
    # with the initial data, there is a need 
    # for a unique index...

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    # this time the output is csv, not json
    # note that read_csv requires a file-like object
    # and we have a str, so io.StringIO comes to the rescue
    result = pd.read_csv(io.StringIO(response.text))

    return result


In [10]:
# votre code

def localize_many(filename, col_number, col_type, col_name, col_city):
    df=pd.read_csv(filename,encoding='utf-8')
    masss=mass_post(filename,col_number,col_type,col_name,col_city)
    #On va chercher le résultat de mass_post
    masss=masss[['latitude','longitude','result_city','result_type']]
    #On sélectionne les colonnes que l'on veut
    resultat = pd.merge(masss,addresses,left_index=True,right_index=True)
    #On va fusionner nos coordonnées géographiques et le tableau initial des rues.
    return resultat

In [11]:
# try your code on the small sample for starters

addresses_small = localize_many("data/addresses-small.csv", "number", "type", "name", "city")
addresses_small


,latitude,longitude,result_city,result_type,number,type,name,city
0,48.888134,2.376448,Paris,housenumber,5,rue,RIQUET,Paris
1,46.669103,-1.418872,La Roche-sur-Yon,street,141,boulevard,ITALIE,Paris
2,48.876160,2.302766,Paris,housenumber,234,rue,FAUBOURG ST-HONORÉ,Paris
3,48.835423,2.402040,Paris,housenumber,62,rue,MICHEL BIZOT,Paris
4,48.863311,2.351775,Paris,housenumber,6,rue,OURS,Paris
5,48.852945,2.364722,Paris,housenumber,30,rue,PETIT-MUSC,Paris
6,48.895498,2.351580,Paris,housenumber,3,impasse,MASSONET,Paris
7,48.867023,2.388347,Paris,housenumber,56,rue,PANOYAUX,Paris
8,48.866374,2.335212,Paris,housenumber,16,rue,THÉRÈSE,Paris
9,48.889520,2.358170,Paris,housenumber,11,rue,JEAN ROBERT,Paris


In [12]:
# sanity check : look at the entries that have
# result_city != 'Paris'
# (or result_type != 'housenumber')
# and decide what to do with them

addresses_small[(addresses_small.result_city == 'Paris')&(addresses_small.result_type == 'housenumber')]

,latitude,longitude,result_city,result_type,number,type,name,city
0,48.888134,2.376448,Paris,housenumber,5,rue,RIQUET,Paris
2,48.876160,2.302766,Paris,housenumber,234,rue,FAUBOURG ST-HONORÉ,Paris
3,48.835423,2.402040,Paris,housenumber,62,rue,MICHEL BIZOT,Paris
4,48.863311,2.351775,Paris,housenumber,6,rue,OURS,Paris
5,48.852945,2.364722,Paris,housenumber,30,rue,PETIT-MUSC,Paris
6,48.895498,2.351580,Paris,housenumber,3,impasse,MASSONET,Paris
7,48.867023,2.388347,Paris,housenumber,56,rue,PANOYAUX,Paris
8,48.866374,2.335212,Paris,housenumber,16,rue,THÉRÈSE,Paris
9,48.889520,2.358170,Paris,housenumber,11,rue,JEAN ROBERT,Paris


In [13]:

#addresses = localize_many("data/addresses.csv", "number", "type", "name", "city")
#On fait notre gros appel pour aller chercher les coordonées géographiques
#addresses.to_csv('data/addresses-complete.csv',index=False)
#On l'enregistre dans le fichier addresses-complete
#Maintenant que l'on a fait cet appel, ce n'est plus la peine de le refaire, d'où les commentaires
#devant ces appels.

In [14]:
# sanity check
len(addresses)


20897

In [17]:
# ~ chatelet

CENTER = 48.856542, 2.347614


In [15]:
import folium
# pour commencer on va recharger la dataframe précédente
import pandas as pd

addresses = pd.read_csv("data/addresses-complete.csv")

# et en faire un petit échantillon

addresses_small = addresses.iloc[:20]


In [19]:
# créez une map centrée sur ce point et de zoom 13
# n'oubliez pas de l'afficher,
# et vérifiez que vous voyez bien Paris, 
# que vous pouvez zoomer et vous déplacer, ...

# votre code
def paris_map():
    return(folium.Map(location=CENTER,zoom_start=13))
# pour l'afficher
paris_map()

On ajoute les adresses

In [ ]:
# your code here

def map_addresses(geoloc):
    """
    creates folium map centered on Paris, with the various addresses
    contained in the input dataframe shown as a marker
    """
    carte=paris_map()
    for _, row in geoloc.iterrows():
        q

In [22]:
# and try it out

# make sure you use A SMALL DATAFRAME because with this method
# trying to display tens of thousands addresses
# is going to be SUPER SLOOOOOW !

map_addresses(addresses_small)


TypeError: Map.__init__() got multiple values for argument 'location'